# Pipeline Xóa Tóc & Ghép Tóc Mới (Hair Removal + Overlay)

Notebook này tái hiện **đúng từng bước** quy trình xử lý ảnh khi người dùng thực hiện try-on qua API `POST /tryon`.

**File nguồn trong dự án:**
- `backend/app/api/tryon.py` — Endpoint điều phối, query Supabase DB + download Storage
- `backend/app/database/db.py` — Supabase client (singleton)
- `backend/app/services/hair_remover.py` — `remove_old_hair()`
- `backend/app/services/hair_overlay.py` — `overlay_hair()`
- `backend/app/services/hair_segmenter.py` — `HairSegmenter`
- `backend/app/services/mediapipe_models.py` — Face Landmarker

---
## Tổng quan Pipeline (bao gồm Supabase)

```
User gọi POST /tryon (person_image + hairstyle_id)
         ↓
[DB] supabase.table("hairstyles").select("image_path").eq("id", hairstyle_id)
         ↓  public_url (Supabase Storage CDN)
[Storage] urllib.request.urlopen(public_url) → RGBA bytes
         ↓
════════════ PHASE 1: XÓA TÓC CŨ ════════════
[B1]  HairSegmenter → hair_mask
[B2]  Dilate + HSV dark → erase_mask
[B3]  Sample skin_ref (outer ring)
[B4]  Pre-fill + TELEA Inpaint (21px)
[B5]  Distance-Weighted Color Correction
[B6]  Gaussian Smoothing
[B7]  Pass-2 Cleanup
[B8]  Ear Restore (landmark 234, 454)
[B9]  Feather Blend
         ↓  no_hair_pil
════════════ PHASE 2: GHÉP TÓC MỚI ══════════
[B10] Face Geometry (468 landmarks)
[B11] Face-Anchored Placement
[B12] Hairline Alpha Mask (polynomial bậc 2)
[B13] Affine Transform (warpAffine)
[B14] Composite + Blend Hairline
         ↓
StreamingResponse (PNG bytes) → Frontend
  ↑ KHÔNG lưu lại trên server (privacy-first)
```

## Cài đặt & Import

In [ ]:
# !pip install mediapipe opencv-python-headless pillow numpy matplotlib supabase

import io
import os
import urllib.request
import warnings
warnings.filterwarnings('ignore')

import cv2
import mediapipe as mp
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print(f"✓ MediaPipe: {mp.__version__}  |  OpenCV: {cv2.__version__}")

## Tải mô hình MediaPipe

Pipeline dùng 2 model từ Google MediaPipe:
1. **Hair Segmenter** (`hair_segmenter.tflite`, ~3MB) — tách vùng tóc
2. **Face Landmarker** (`face_landmarker.task`, ~12MB) — 468 điểm mốc khuôn mặt

**File:** `mediapipe_models.py`

In [ ]:
MODELS_DIR = os.environ.get("MODELS_DIR", ".")
os.makedirs(MODELS_DIR, exist_ok=True)

_MODELS = {
    "hair_segmenter.tflite": (
        "https://storage.googleapis.com/mediapipe-models/"
        "image_segmenter/hair_segmenter/float32/latest/hair_segmenter.tflite"
    ),
    "face_landmarker.task": (
        "https://storage.googleapis.com/mediapipe-models/"
        "face_landmarker/face_landmarker/float16/latest/face_landmarker.task"
    ),
}

for fname, url in _MODELS.items():
    fpath = os.path.join(MODELS_DIR, fname)
    if not os.path.exists(fpath):
        print(f"Đang tải {fname}...")
        urllib.request.urlretrieve(url, fpath)
        print(f"  ✓ Xong")
    else:
        print(f"✓ {fname} đã có")

hair_seg_options = vision.ImageSegmenterOptions(
    base_options=python.BaseOptions(
        model_asset_path=os.path.join(MODELS_DIR, "hair_segmenter.tflite")
    ),
    output_confidence_masks=True,
)
hair_segmenter = vision.ImageSegmenter.create_from_options(hair_seg_options)

lm_options = vision.FaceLandmarkerOptions(
    base_options=python.BaseOptions(
        model_asset_path=os.path.join(MODELS_DIR, "face_landmarker.task")
    ),
    num_faces=1,
)
face_landmarker = vision.FaceLandmarker.create_from_options(lm_options)
print("\n✓ Tất cả model sẵn sàng")

---
## Bước 0 — Query Supabase DB + Download ảnh tóc từ Storage

**Công nghệ:** Supabase PostgreSQL + Supabase Storage CDN

Đây là bước đầu tiên trong `tryon.py` trước khi chạy bất kỳ ML nào:

**Luồng thực tế trong `tryon.py`:**
```python
# 1. Query Supabase DB lấy image_path (CDN URL)
result = (
    supabase.table("hairstyles")
    .select("image_path")
    .eq("id", hairstyle_id)
    .maybe_single()
    .execute()
)
public_url = result.data["image_path"]
# → "https://xxx.supabase.co/storage/v1/object/public/hairstyles/toc_ngan_a3b2.png"

# 2. Download RGBA PNG từ Supabase Storage (CDN)
with urllib.request.urlopen(public_url, timeout=10) as resp:
    hair_bytes = resp.read()

hair_pil = Image.open(io.BytesIO(hair_bytes)).convert("RGBA")
```

**Tại sao dùng `urllib.request` thay vì Supabase SDK để download?**
- `image_path` đã là **public CDN URL** → không cần auth
- `urllib.request` có sẵn trong Python stdlib → không thêm dependency
- CDN của Supabase (Cloudflare) phục vụ ảnh nhanh hơn backend server

In [ ]:
# ── Mô phỏng Supabase DB Query ────────────────────────────────────────────────
# Trong production, thay bằng:
#   from supabase import create_client
#   supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
#   result = supabase.table("hairstyles").select("image_path").eq("id", hairstyle_id).maybe_single().execute()
#   public_url = result.data["image_path"]

HAIRSTYLE_ID = 1  # ID kiểu tóc trong Supabase DB

# Ở notebook này, load trực tiếp từ file local để demo
HAIR_IMAGE_PATH   = "backend/assets/hairstyles/t2.png"   # THAY bằng ảnh RGBA tóc của bạn
USER_IMAGE_PATH   = "backend/assets/hairstyles/t1.png"   # THAY bằng ảnh chân dung của bạn

print("═" * 55)
print("  SUPABASE DB QUERY (mô phỏng)")
print("═" * 55)
simulated_db_row = {
    "id"         : HAIRSTYLE_ID,
    "name"       : "Tóc Ngắn Layer",
    "image_path" : "https://abcxyz.supabase.co/storage/v1/object/public/hairstyles/toc_ngan_layer_a3b2c1d4.png",
    "created_at" : "2025-06-03T10:00:00+00:00",
}
for k, v in simulated_db_row.items():
    print(f"  {k:<15}: {v}")

print("\n  → public_url = image_path")
print("  → urllib.request.urlopen(public_url) → RGBA PNG bytes")
print("  → Image.open(BytesIO(hair_bytes)).convert('RGBA')")

# Load ảnh cho demo
_MAX_DIM = 800
person_pil = Image.open(USER_IMAGE_PATH).convert("RGBA")
if max(person_pil.size) > _MAX_DIM:
    person_pil.thumbnail((_MAX_DIM, _MAX_DIM), Image.LANCZOS)
hair_pil = Image.open(HAIR_IMAGE_PATH).convert("RGBA")

person_rgb = np.array(person_pil.convert("RGB"))
hair_rgba  = np.array(hair_pil)
print(f"\n✓ Ảnh người dùng: {person_pil.size}, mode={person_pil.mode}")
print(f"✓ Kiểu tóc:        {hair_pil.size}, mode={hair_pil.mode}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(person_rgb)
axes[0].set_title(f"Ảnh người dùng\n(POST /tryon body)", fontsize=12)
axes[0].axis('off')
bg = np.ones((*hair_rgba.shape[:2], 3), dtype=np.uint8) * 200
a  = hair_rgba[:,:,3:4].astype(np.float32) / 255
axes[1].imshow((hair_rgba[:,:,:3] * a + bg * (1-a)).astype(np.uint8))
axes[1].set_title(f"Kiểu tóc\n(từ Supabase Storage CDN)", fontsize=12)
axes[1].axis('off')
plt.suptitle("Bước 0: DB Query + Storage Download", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# PHASE 1: XÓA TÓC CŨ

---
## Bước 1 — Lấy Hair Mask từ HairSegmenter

**File:** `hair_remover.py` lines 76–77 (chi tiết xem `hair.ipynb`)

In [ ]:
def get_hair_mask_from_pil(pil_img):
    buf = io.BytesIO()
    pil_img.convert("RGB").save(buf, format="PNG")
    img_rgb = np.array(Image.open(io.BytesIO(buf.getvalue())).convert("RGB"))
    mp_img  = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result  = hair_segmenter.segment(mp_img)
    conf    = np.squeeze(np.array(result.confidence_masks[1].numpy_view(), dtype=np.float32))
    k9 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    core = (conf > 0.50).astype(np.uint8) * 255
    core = cv2.morphologyEx(core, cv2.MORPH_CLOSE, k9, iterations=2)
    core = cv2.morphologyEx(core, cv2.MORPH_OPEN, k9)
    k3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    raw = (conf > 0.20).astype(np.uint8) * 255
    raw = cv2.morphologyEx(raw, cv2.MORPH_CLOSE, k3)
    _, labels = cv2.connectedComponents(cv2.bitwise_or(raw, core))
    keep = np.zeros_like(raw)
    for lbl in set(int(v) for v in np.unique(labels[core > 0]) if v != 0):
        keep[labels == lbl] = 255
    strands = cv2.bitwise_and(keep, cv2.bitwise_and(raw, cv2.bitwise_not(core)))
    return cv2.bitwise_or(core, strands), conf

hair_mask, conf_map = get_hair_mask_from_pil(person_pil)
img_rgb = np.array(person_pil.convert("RGB"))
H, W    = img_rgb.shape[:2]
print(f"Hair mask: {hair_mask.shape}, tóc: {(hair_mask>0).sum():,} px")

ov = img_rgb.copy()
ov[hair_mask>0] = (ov[hair_mask>0].astype(np.float32)*0.4 + np.array([255,50,50])*0.6).astype(np.uint8)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_rgb); axes[0].set_title("Ảnh gốc"); axes[0].axis('off')
axes[1].imshow(conf_map, cmap='hot', vmin=0, vmax=1); axes[1].set_title("Confidence Map"); axes[1].axis('off')
axes[2].imshow(ov); axes[2].set_title("Hair Mask (đỏ)"); axes[2].axis('off')
plt.suptitle("Bước 1: Hair Mask từ MediaPipe", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 2 — Tạo Erase Mask

**Công nghệ:** Morphological Dilation + HSV Dark Pixel Filter

- `mask_dilate = dilate(hair_mask, 13×13, ×3)` — mở rộng vùng xóa
- `dark = HSV(V<80) ∩ dilate(hair_mask, ×4)` — bắt sợi tóc tối sót
- `erase_mask = mask_dilate | dark`

**File:** `hair_remover.py` lines 79–88

In [ ]:
k_big = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13))
k_med = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
mask_dilate = cv2.dilate(hair_mask, k_big, iterations=3)
hsv  = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
dark = cv2.inRange(hsv, np.array([0,0,0]), np.array([180,255,80]))
dark = cv2.bitwise_and(dark, cv2.dilate(hair_mask, k_big, iterations=4))
erase_mask = cv2.bitwise_or(mask_dilate, dark)
eys, exs   = np.where(erase_mask > 0)

print(f"Hair: {(hair_mask>0).sum():,} | Dilated: {(mask_dilate>0).sum():,} | Dark: {(dark>0).sum():,} | Erase: {(erase_mask>0).sum():,}")

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, img, title in zip(axes, [hair_mask, mask_dilate, dark, erase_mask],
                           ["Hair Mask", "Dilate ×3", "Dark (V<80)", "Erase Mask"]):
    ax.imshow(img, cmap='gray'); ax.set_title(title, fontsize=11); ax.axis('off')
plt.suptitle("Bước 2: Erase Mask", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 3 — Sample màu da tham chiếu

**File:** `hair_remover.py` → `_sample_skin_ref()` lines 23–37

In [ ]:
k_in  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
k_out = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (30, 30))
outer = cv2.bitwise_and(
    cv2.dilate(erase_mask, k_out),
    cv2.bitwise_not(cv2.dilate(erase_mask, k_in))
)
ys_o, xs_o = np.where(outer > 0)
colors = img_rgb[ys_o, xs_o].astype(np.float32)
bright = colors.mean(axis=1) > 80
skin_ref = colors[bright].mean(axis=0) if bright.sum() >= 5 else np.array([210,175,150], dtype=np.float32)
print(f"skin_ref (RGB): {skin_ref.astype(int)}")

ov2 = img_rgb.copy()
ov2[outer>0] = (ov2[outer>0].astype(np.float32)*0.4 + np.array([0,255,100])*0.6).astype(np.uint8)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img_rgb); axes[0].set_title("Ảnh gốc"); axes[0].axis('off')
axes[1].imshow(ov2); axes[1].set_title("Outer ring (lấy màu da)"); axes[1].axis('off')
axes[2].imshow(np.ones((60,120,3),np.uint8)*skin_ref.astype(np.uint8))
axes[2].set_title(f"skin_ref RGB({skin_ref[0]:.0f},{skin_ref[1]:.0f},{skin_ref[2]:.0f})")
axes[2].axis('off')
plt.suptitle("Bước 3: Sample màu da", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 4 — Pre-fill + OpenCV TELEA Inpaint

**Công nghệ:** `cv2.inpaint(img_seeded, erase_mask, 21, cv2.INPAINT_TELEA)`

- **Pre-fill** màu da → TELEA không bị kéo về màu tối của tóc
- **TELEA** lan truyền từ biên mask vào trong, trọng số theo khoảng cách

**File:** `hair_remover.py` lines 90–95

In [ ]:
img_seeded = img_rgb.copy()
img_seeded[eys, exs] = skin_ref.astype(np.uint8)
inpainted = cv2.inpaint(img_seeded, erase_mask, 21, cv2.INPAINT_TELEA)

print(f"Inpaint: {len(eys):,} pixel được điền lại")
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes, [img_rgb, img_seeded, inpainted],
                           ["Ảnh gốc", "Pre-fill", "TELEA Inpaint"]):
    ax.imshow(img); ax.set_title(title, fontsize=12); ax.axis('off')
plt.suptitle("Bước 4: Pre-fill + TELEA Inpaint", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 5 — Distance-Weighted Color Correction

**File:** `hair_remover.py` lines 97–118

```
dist_norm = (distanceTransform / d_max) ^ 0.6
color_delta = (skin_ref - inpainted_mean) × 0.45
offset = dist_norm × color_delta
```

In [ ]:
dist  = cv2.distanceTransform(erase_mask, cv2.DIST_L2, 5).astype(np.float32)
d_max = dist.max()
inpainted_corrected = inpainted.copy()

if d_max > 0:
    dist_norm   = (dist / d_max) ** 0.6
    inp_mean    = inpainted[eys, exs].astype(np.float32).mean(axis=0)
    color_delta = (skin_ref - inp_mean) * 0.45
    offset_map  = dist_norm[:,:,None] * color_delta[None,None,:]
    inp_f = inpainted.astype(np.float32)
    inp_f[eys, exs] = (inp_f[eys, exs] + offset_map[eys, exs]).clip(0, 255)
    inpainted_corrected = inp_f.astype(np.uint8)

print(f"dist_max: {d_max:.1f}px | delta: {color_delta.astype(int)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
im = axes[0].imshow(dist, cmap='plasma')
plt.colorbar(im, ax=axes[0], fraction=0.046)
axes[0].set_title("Distance Transform"); axes[0].axis('off')
axes[1].imshow(inpainted); axes[1].set_title("Trước"); axes[1].axis('off')
axes[2].imshow(inpainted_corrected); axes[2].set_title("Sau Color Correction"); axes[2].axis('off')
plt.suptitle("Bước 5: Distance-Weighted Color Correction", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 6 — Gaussian Smoothing

**File:** `hair_remover.py` lines 111–118

`result = inpainted × (1 - sw) + blur × sw` với `sw = dist_norm × 0.35`

In [ ]:
smoothed = cv2.GaussianBlur(inpainted_corrected, (11, 11), 0)
sw = (dist_norm * 0.35)[:,:,None]
inp_f2 = inpainted_corrected.astype(np.float32)
inp_f2[eys, exs] = (
    inp_f2[eys, exs] * (1 - sw[eys, exs])
    + smoothed.astype(np.float32)[eys, exs] * sw[eys, exs]
).clip(0, 255)
after_smooth = inp_f2.astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes, [smoothed, inpainted_corrected, after_smooth],
                           ["Gaussian blur 11×11", "Trước smooth", "Sau smooth"]):
    ax.imshow(img); ax.set_title(title, fontsize=12); ax.axis('off')
plt.suptitle("Bước 6: Gaussian Smoothing", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 7 — Pass-2 Cleanup

**File:** `hair_remover.py` lines 120–127

Phát hiện pixel tối sót (V < 90) trong edge band → inpaint lần 2 (kernel 15px)

In [ ]:
edge_band = cv2.dilate(hair_mask, k_big, iterations=4)
dark2     = cv2.inRange(cv2.cvtColor(after_smooth, cv2.COLOR_RGB2HSV),
                        np.array([0,0,0]), np.array([180,255,90]))
residual  = cv2.bitwise_and(dark2, edge_band)
after_cleanup = after_smooth.copy()
if residual.sum() > 0:
    after_cleanup = cv2.inpaint(after_smooth, cv2.dilate(residual, k_med, 1), 15, cv2.INPAINT_TELEA)
    print(f"Residual: {(residual>0).sum():,} px → inpaint lần 2")
else:
    print("Pass-2: không có tóc sót")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes, [residual, after_smooth, after_cleanup],
                           ["Residual dark", "Trước pass-2", "Sau pass-2"]):
    ax.imshow(img, cmap='gray' if img.ndim==2 else None); ax.set_title(title, fontsize=12); ax.axis('off')
plt.suptitle("Bước 7: Pass-2 Cleanup", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 8 — Khôi phục tai (Ear Restore)

**Công nghệ:** Face Landmarker landmark `234` (tai trái) + `454` (tai phải) + Feather Blend

**File:** `hair_remover.py` → `_make_ear_restore_mask()` lines 40–62

In [ ]:
mp_img_ear = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
ear_result = face_landmarker.detect(mp_img_ear)
ear_mask   = np.zeros((H, W), np.uint8)
after_ear  = after_cleanup.copy()

if ear_result.face_landmarks:
    lm   = ear_result.face_landmarks[0]
    lc_x = int(lm[234].x * W); lc_y = int(lm[234].y * H)
    rc_x = int(lm[454].x * W); rc_y = int(lm[454].y * H)
    fw   = max(1, abs(rc_x - lc_x))
    er_w = int(fw * 0.10); er_h = int(fw * 0.19); off = int(er_w * 0.5)
    cv2.ellipse(ear_mask, (lc_x-off, lc_y), (er_w, er_h), 0, 0, 360, 255, -1)
    cv2.ellipse(ear_mask, (rc_x+off, rc_y), (er_w, er_h), 0, 0, 360, 255, -1)
    ear_alpha = cv2.GaussianBlur(ear_mask.astype(np.float32), (7,7), 0) / 255.0
    after_ear = (
        after_cleanup.astype(np.float32) * (1 - ear_alpha[:,:,None])
        + img_rgb.astype(np.float32) * ear_alpha[:,:,None]
    ).clip(0, 255).astype(np.uint8)
    print(f"Ear left=({lc_x},{lc_y}), right=({rc_x},{rc_y}), fw={fw}px")

vis_ear = img_rgb.copy()
vis_ear[ear_mask>0] = (vis_ear[ear_mask>0].astype(np.float32)*0.4 + np.array([0,200,255])*0.6).astype(np.uint8)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes, [vis_ear, after_cleanup, after_ear],
                           ["Ear mask (xanh)", "Trước", "Sau ear restore"]):
    ax.imshow(img); ax.set_title(title, fontsize=12); ax.axis('off')
plt.suptitle("Bước 8: Khôi phục tai", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 9 — Feather Blend → Ảnh không tóc

**File:** `hair_remover.py` lines 142–156

`result = inpainted × blur_mask + original × (1 - blur_mask)`

In [ ]:
blur_mask   = cv2.GaussianBlur(erase_mask.astype(np.float32), (11, 11), 0) / 255.0
result_rgb  = (
    after_ear.astype(np.float32) * blur_mask[:,:,None]
    + img_rgb.astype(np.float32) * (1 - blur_mask[:,:,None])
).clip(0, 255).astype(np.uint8)

rgba_nohair = np.dstack([result_rgb, np.full((H, W), 255, np.uint8)])
no_hair_pil = Image.fromarray(rgba_nohair, "RGBA")

print(f"no_hair_pil: {no_hair_pil.size}, mode={no_hair_pil.mode}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes, [img_rgb, blur_mask, result_rgb],
                           ["Ảnh gốc", "Feather Mask", "no_hair (RGBA)"]):
    ax.imshow(img, cmap='gray' if img.ndim==2 else None); ax.set_title(title, fontsize=12); ax.axis('off')
plt.suptitle("Bước 9: Feather Blend → Ảnh không tóc", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
# PHASE 2: GHÉP TÓC MỚI

---
## Bước 10 — Face Geometry (468 Landmarks)

**File:** `hair_overlay.py` → `_face_geometry()` lines 16–82

In [ ]:
def lm_pt(lm, idx, W, H):
    return np.array([lm[idx].x*W, lm[idx].y*H], dtype=np.float64)

def face_geometry(img_rgb_arr):
    H, W = img_rgb_arr.shape[:2]
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb_arr)
    res = face_landmarker.detect(mp_img)
    if not res.face_landmarks: return None
    lm = res.face_landmarks[0]
    def pt(i): return lm_pt(lm, i, W, H)
    left_cheek = pt(234); right_cheek = pt(454)
    face_width = float(np.linalg.norm(pt(356)-pt(127)))
    temple_left = pt(21); temple_right = pt(251)
    le = np.array([pt(i) for i in [33,133,159,145]]).mean(0)
    re = np.array([pt(i) for i in [362,263,386,374]]).mean(0)
    ec = (le+re)*0.5; ev = re-le
    ang = float(np.degrees(np.arctan2(ev[1], ev[0])))
    brow_y = float(np.mean([pt(i)[1] for i in [70,63,105,66,107,336,296,334,293,300]]))
    nose_tip_y = float(pt(4)[1])
    hairline_pt = np.array([ec[0], 2.0*brow_y-nose_tip_y])
    nose_x = float(pt(4)[0])
    yaw_ratio = abs(nose_x-left_cheek[0]) / max(1.0, abs(right_cheek[0]-nose_x))
    return dict(lm=lm, W=W, H=H, face_width=face_width,
                temple_left=temple_left, temple_right=temple_right,
                angle_deg=ang, eye_center=ec, brow_y=brow_y,
                nose_tip_y=nose_tip_y, hairline_pt=hairline_pt,
                yaw_ratio=yaw_ratio, left_cheek=left_cheek, right_cheek=right_cheek,
                lm10=pt(10))

geo = face_geometry(img_rgb)
if geo is None: raise RuntimeError("Không phát hiện khuôn mặt!")

print(f"face_width={geo['face_width']:.1f}px | angle={geo['angle_deg']:.2f}° | yaw={geo['yaw_ratio']:.3f}")
print(f"brow_y={geo['brow_y']:.1f} | hairline=({geo['hairline_pt'][0]:.1f},{geo['hairline_pt'][1]:.1f})")

vis_lm = img_rgb.copy()
lm = geo['lm']
for idx, c in [(234,(255,80,80)),(454,(255,80,80)),(21,(0,200,255)),(251,(0,200,255)),
                (10,(0,255,100)),(4,(255,200,0)),(70,(255,200,0)),(336,(255,200,0))]:
    cv2.circle(vis_lm, (int(lm[idx].x*W), int(lm[idx].y*H)), 6, c, -1)
cv2.circle(vis_lm, tuple(geo['hairline_pt'].astype(int)), 10, (255,0,255), -1)
cv2.circle(vis_lm, tuple(geo['eye_center'].astype(int)),   10, (0,255,0),  -1)
plt.figure(figsize=(6, 6))
plt.imshow(vis_lm)
plt.title("Bước 10: Face Landmarks\n● magenta=hairline ● green=eye_center", fontsize=12)
plt.axis('off'); plt.tight_layout(); plt.show()

---
## Bước 11 — Face-Anchored Placement

**File:** `hair_overlay.py` → `_face_anchored_placement()` lines 179–242

In [ ]:
hair_rgba_arr = np.array(hair_pil.convert("RGBA"))
ys_h, xs_h   = np.where(hair_rgba_arr[:,:,3] > 5)
if len(ys_h):
    hair_rgba_arr = hair_rgba_arr[ys_h.min():ys_h.max()+1, xs_h.min():xs_h.max()+1]
alpha_h = hair_rgba_arr[:,:,3]
ys_h, xs_h = np.where(alpha_h > 10)
bbox_w = float(xs_h.max()-xs_h.min()); hair_top_r = float(ys_h.min())
h_h, w_h = hair_rgba_arr.shape[:2]

# Mask geometry
lx = max(0, int(geo['left_cheek'][0])); rx = min(W-1, int(geo['right_cheek'][0]))
root_y = None
for row in range(int(geo['brow_y']), -1, -1):
    if float((hair_mask[row, lx:rx]>0).sum()) / max(rx-lx,1) >= 0.05:
        root_y = row; break
oys, oxs = np.where(hair_mask > 0)
mask_cx = (float(oxs.min())+float(oxs.max()))/2 if len(oxs)>=20 else W/2
mask_width = float(oxs.max()-oxs.min()) if len(oxs)>=20 else geo['face_width']
mask_top = float(oys.min()) if len(oys)>=20 else 0
mask_height = max(1.0, float(root_y)-mask_top) if root_y else geo['brow_y']

if root_y:
    target_y = float(root_y); target_x = float(mask_cx)
    scale_w  = mask_width / max(bbox_w, 1)
    scale_h  = mask_height / max(1, float(np.where(alpha_h>10)[0].max())-hair_top_r)
    scale    = float(np.clip(max(scale_w, scale_h), 0.3, 2.5))
else:
    hp = geo['hairline_pt']; target_y = float(hp[1]); target_x = float(hp[0])
    scale = float(np.clip(geo['face_width']/max(bbox_w,1), 0.3, 2.5))

cy_hair = int(ys_h.max())
for row in range(h_h-1, -1, -1):
    if float((alpha_h[row]>10).sum())/max(w_h,1) >= 0.15:
        cy_hair = row; break
skew    = np.clip((geo['yaw_ratio']-1.0)*0.12, -0.06, 0.06)
cx_hair = int(float(xs_h.min()) + bbox_w*(0.5+skew))

print(f"Scale={scale:.3f}x | target=({target_x:.0f},{target_y:.0f}) | anchor_hair=({cx_hair},{cy_hair})")
print(f"yaw_ratio={geo['yaw_ratio']:.3f} → skew={skew:.4f}")

vis_p = result_rgb.copy()
if root_y: cv2.circle(vis_p, (int(mask_cx), int(root_y)), 10, (255,0,0), -1)
cv2.circle(vis_p, tuple(geo['hairline_pt'].astype(int)), 8, (255,0,255), -1)
bg_vis = np.ones((*hair_rgba_arr.shape[:2],3), np.uint8)*220
a_v = hair_rgba_arr[:,:,3:4].astype(np.float32)/255
h_vis2 = (hair_rgba_arr[:,:,:3]*a_v + bg_vis*(1-a_v)).astype(np.uint8)
cv2.circle(h_vis2, (cx_hair, cy_hair), 10, (255,0,0), -1)
fig, axes = plt.subplots(1,2,figsize=(12,5))
axes[0].imshow(vis_p); axes[0].set_title("● đỏ=target | ● magenta=hairline"); axes[0].axis('off')
axes[1].imshow(h_vis2); axes[1].set_title(f"● đỏ=anchor_hair | scale={scale:.2f}x"); axes[1].axis('off')
plt.suptitle("Bước 11: Face-Anchored Placement", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 12 — Hairline Alpha Mask (Polynomial bậc 2)

**File:** `hair_overlay.py` → `_build_hairline_alpha()` lines 117–159

In [ ]:
def build_hairline_alpha(geo, H, W, feather_below=12, hairline_y=None):
    lm = geo['lm']; lW, lH = geo['W'], geo['H']
    pts = np.array([[lm[i].x*lW, lm[i].y*lH] for i in [10,109,67,103,54,21,162,338,297,332,284,251,389]])
    pts = pts[np.argsort(pts[:,0])]
    coeffs = np.polyfit(pts[:,0], pts[:,1], 2)
    xs_a = np.arange(W, dtype=np.float32)
    hl_y = np.polyval(coeffs, xs_a)
    if hairline_y is not None:
        cx = float(geo['eye_center'][0])
        hl_y = hl_y + (hairline_y - float(np.polyval(coeffs, cx)))
    y_grid = np.arange(H, dtype=np.float32)[:,None]
    dist   = hl_y[None,:] - y_grid
    alpha  = np.where(dist>=0, np.ones_like(dist),
                      np.clip(1.0+dist/float(feather_below), 0.0, 1.0)).astype(np.float32)
    tx_l = float(geo['temple_left'][0]); tx_r = float(geo['temple_right'][0])
    fade_w = float(tx_r-tx_l)*0.12
    weight = np.ones(W, np.float32)
    for x in range(W):
        if x < tx_l: weight[x] = max(0.0, (x-(tx_l-fade_w))/fade_w) if fade_w>0 else 0.0
        elif x > tx_r: weight[x] = max(0.0, ((tx_r+fade_w)-x)/fade_w) if fade_w>0 else 0.0
    return (alpha*weight[None,:] + 1.0*(1.0-weight[None,:])).astype(np.float32), hl_y, pts

hl_alpha, hl_y_curve, hl_pts = build_hairline_alpha(geo, H, W, feather_below=12, hairline_y=target_y)
print(f"hl_alpha: {hl_alpha.shape}, range=[{hl_alpha.min():.2f},{hl_alpha.max():.2f}]")

fig, axes = plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(img_rgb); axes[0].plot(np.arange(W), hl_y_curve, 'y-', lw=3, label='Hairline poly-2')
axes[0].scatter(hl_pts[:,0], hl_pts[:,1], c='red', s=60, zorder=5, label='Landmarks')
axes[0].legend(fontsize=9); axes[0].set_title("Polynomial Fit"); axes[0].axis('off')
axes[1].imshow(hl_alpha, cmap='gray', vmin=0, vmax=1)
axes[1].plot(np.arange(W), hl_y_curve, 'r-', lw=2); axes[1].set_title("Hairline Alpha"); axes[1].axis('off')
cx_p = int(W//2)
axes[2].plot(hl_alpha[:,cx_p], np.arange(H), 'b-', lw=2)
axes[2].axhline(hl_y_curve[cx_p], color='r', ls='--', label='hairline_y')
axes[2].set(xlabel='Alpha', ylabel='Y px', title='Cross-section'); axes[2].invert_yaxis()
axes[2].legend(); axes[2].grid(alpha=0.3)
plt.suptitle("Bước 12: Hairline Alpha Mask", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 13 — Affine Transform (warpAffine)

**File:** `hair_overlay.py` → `_similarity_matrix()` + `overlay_hair()` lines 245–357

```
M = | scale·cos(θ)  -scale·sin(θ)  tx |
    | scale·sin(θ)   scale·cos(θ)  ty |
```

In [ ]:
theta = np.radians(geo['angle_deg']); c, s = np.cos(theta), np.sin(theta)
ax_, ay_ = float(cx_hair), float(cy_hair)
dx_, dy_ = float(target_x), float(target_y)
M = np.float32([
    [scale*c, -scale*s, -scale*c*ax_+scale*s*ay_+dx_],
    [scale*s,  scale*c, -scale*s*ax_-scale*c*ay_+dy_],
])
print(f"M: scale={scale:.3f}x, angle={geo['angle_deg']:.2f}°")
print(f"  | {M[0,0]:+.4f}  {M[0,1]:+.4f}  {M[0,2]:+.1f} |")
print(f"  | {M[1,0]:+.4f}  {M[1,1]:+.4f}  {M[1,2]:+.1f} |")

wa = cv2.warpAffine(hair_rgba_arr[:,:,:3], M, (W,H), flags=cv2.INTER_CUBIC,
                    borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0))
walpha_raw = cv2.warpAffine(hair_rgba_arr[:,:,3], M, (W,H), flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_CONSTANT, borderValue=0)
walpha_raw = np.clip((walpha_raw.astype(np.float32)/255.0)**0.95*255, 0, 255).astype(np.uint8)
walpha = (walpha_raw.astype(np.float32)*hl_alpha).clip(0,255).astype(np.uint8)
warped = np.dstack([wa, walpha])

bg_g = np.ones((H,W,3),np.uint8)*200
a_w  = walpha[:,:,None].astype(np.float32)/255
wa_vis = (wa.astype(np.float32)*a_w + bg_g.astype(np.float32)*(1-a_w)).astype(np.uint8)
fig, axes = plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(wa_vis); axes[0].set_title("Tóc sau Affine Warp"); axes[0].axis('off')
axes[1].imshow(walpha_raw, cmap='gray'); axes[1].set_title("Alpha trước hl_mask"); axes[1].axis('off')
axes[2].imshow(walpha,     cmap='gray'); axes[2].set_title("Alpha sau hl_mask");   axes[2].axis('off')
plt.suptitle("Bước 13: Affine Transformation", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 14 — Composite + Blend Hairline → Kết quả cuối

**File:** `hair_overlay.py` → `_composite()` + `_blend_hairline()` lines 258–299

In [ ]:
# Composite (Porter-Duff source-over)
base_rgb  = np.array(no_hair_pil.convert("RGB"))
H2, W2   = base_rgb.shape[:2]
base_rgba = np.dstack([base_rgb, np.full((H2,W2),255,np.uint8)])
a_comp    = warped[:,:,3:4].astype(np.float32)/255
comp_rgb  = (base_rgba[:,:,:3].astype(np.float32)*(1-a_comp) + warped[:,:,:3].astype(np.float32)*a_comp)
comp      = np.dstack([comp_rgb.astype(np.uint8), np.full((H2,W2),255,np.uint8)])

# Blend hairline (morphological edge + shadow)
ha   = warped[:,:,3].astype(np.float32)
edge = cv2.morphologyEx((ha>5).astype(np.uint8)*255, cv2.MORPH_GRADIENT,
                         cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)))
tx_l = int(geo['temple_left'][0]); tx_r = int(geo['temple_right'][0])
y_st = max(0, int(geo['brow_y'] - geo['face_width']*0.25))
zone = np.zeros((H2,W2),np.uint8); zone[y_st:,tx_l:tx_r] = 255
ec_  = cv2.bitwise_and(edge, zone)
f    = cv2.GaussianBlur(ec_,(11,11),0).astype(np.float32)/255
a_ch = ha[:,:,None]/255
blended = warped[:,:,:3].astype(np.float32)*a_ch + base_rgba[:,:,:3].astype(np.float32)*(1-a_ch)
out  = comp[:,:,:3].astype(np.float32)*(1-f[:,:,None]) + blended*f[:,:,None]
shad = cv2.GaussianBlur(ec_,(11,11),0).astype(np.float32)/255
out  = (out*(1-shad[:,:,None]*0.04)).clip(0,255).astype(np.uint8)
final = np.dstack([out, np.full((H2,W2),255,np.uint8)])
final_pil = Image.fromarray(final, "RGBA")

fig, axes = plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(img_rgb); axes[0].set_title("Ảnh gốc"); axes[0].axis('off')
axes[1].imshow(comp[:,:,:3]); axes[1].set_title("Sau Composite"); axes[1].axis('off')
axes[2].imshow(final[:,:,:3]); axes[2].set_title("Kết quả cuối"); axes[2].axis('off')
plt.suptitle("Bước 14: Composite + Blend Hairline", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 15 — StreamingResponse về Frontend (không lưu Supabase)

**Công nghệ:** FastAPI `StreamingResponse` — trả ảnh PNG trực tiếp qua HTTP

**Thiết kế Privacy-First của dự án:**

```
Ảnh người dùng → xử lý trong RAM → trả về → XÓA khỏi bộ nhớ
                                    ↑
                          KHÔNG lưu vào Supabase DB
                          KHÔNG lưu vào Supabase Storage
                          KHÔNG lưu vào disk server
```

**Code trong `tryon.py` (lines 51–56):**
```python
buf = io.BytesIO()
result_pil.convert("RGB").save(buf, format="PNG")
buf.seek(0)
return StreamingResponse(buf, media_type="image/png")
# → PNG bytes stream về frontend, không lưu gì trên server
```

**Frontend (`client.ts`) nhận:**
```typescript
const blob = await res.blob();
return blobToDataUrl(blob);   // → data:image/png;base64,...
// Hiển thị trong Modal Before/After, cho phép Download
```

In [ ]:
# ── Mô phỏng StreamingResponse ────────────────────────────────────────────────
buf = io.BytesIO()
final_pil.convert("RGB").save(buf, format="PNG")
buf.seek(0)
png_bytes = buf.getvalue()

print("═" * 55)
print("  RESPONSE về Frontend")
print("═" * 55)
print(f"  media_type  : image/png")
print(f"  file size   : {len(png_bytes)/1024:.1f} KB")
print(f"  lưu DB?     : KHÔNG")
print(f"  lưu Storage?: KHÔNG")
print(f"  lưu disk?   : KHÔNG")
print(f"  → StreamingResponse trả thẳng PNG bytes về client")

# Tổng kết trực quan 4 bước chính
bg2 = np.ones((H,W,3),np.uint8)*200
a_wv = warped[:,:,3:4].astype(np.float32)/255
wv   = (wa.astype(np.float32)*a_wv + bg2.astype(np.float32)*(1-a_wv)).astype(np.uint8)

fig, axes = plt.subplots(1,4,figsize=(20,5))
panels = [
    (img_rgb,         "① Ảnh gốc\n(người dùng upload)"),
    (result_rgb,      "② Xóa tóc\n(TELEA Inpaint)"),
    (wv,              "③ Tóc warp\n(Supabase Storage → CDN)"),
    (final[:,:,:3],   "④ Kết quả\n(StreamingResponse → Frontend)"),
]
for ax, (img, title) in zip(axes, panels):
    ax.imshow(img); ax.set_title(title, fontsize=12, fontweight='bold'); ax.axis('off')
plt.suptitle("Toàn bộ Pipeline: DB Query → ML → StreamingResponse\n(Ảnh người dùng không lưu trên server)",
             fontsize=14, fontweight='bold', y=1.04)
plt.tight_layout(); plt.show()